## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from langchain_openai import OpenAIEmbeddings

In [48]:
MODEL = "gpt-4.1-mini"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
#embeddings = HuggingFaceEmbeddings(model_name="microsoft/harrier-oss-v1-0.6b",
#                                   model_kwargs={"trust_remote_code": True})
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [79]:
retriever.invoke("Who went to Manchester University?", k=8)

[Document(id='fbc76970-c536-45f9-a04d-122803e53244', metadata={'doc_type': 'contracts', 'source': 'C:\\Users\\ME36352\\MyFolder\\AI\\Repos\\AIRepo\\week5\\knowledge-base\\contracts\\Contract with United Healthcare Alliance for Healthllm.md'}, page_content="3. **Comprehensive Implementation:**\n   - 18-month phased implementation program with 4 major go-live events\n   - Dedicated program management office (PMO) with weekly steering committee\n   - Migration of 250,000+ member records from legacy systems\n   - Integration with 15,000+ provider records and credentialing data\n   - Training for 300+ United staff across all departments\n   - Training for 100+ key provider practices and health systems\n   - Parallel processing with legacy system for 120 days\n   - Go-live support with 12-week on-site Insurellm team presence (6-10 people)\n\n4. **Training and Enablement:**\n   - Comprehensive role-based training programs for all staff\n   - Executive leadership training on healthcare AI and 

In [7]:
llm.invoke("Who is Avery?") 

AIMessage(content='Could you please provide more context or specify which Avery you are referring to? There are many individuals and characters named Avery.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 11, 'total_tokens': 35, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_588e1532e2', 'id': 'chatcmpl-DX6kYbroomV83fHRQ5XrL5OwFiqVu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--5489cfe0-0400-4eef-ac7c-865fb0a8cf6d-0', usage_metadata={'input_tokens': 11, 'output_tokens': 24, 'total_tokens': 35, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Time to put this together!

In [8]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [66]:
def answer_question(question: str, history):
    docs = retriever.invoke(question, k=8)
    print(docs)
    print("\n\n\n\n\n\n")
    print("\n\n111111111111111111111111111111111111111111")
    print(docs[0].page_content)
    print("\n\n222222222222222222222222222222222222222222")
    print(docs[1].page_content)
    print("\n\n333333333333333333333333333333333333333333333")
    print(docs[2].page_content)
    print("\n\n4444444444444444444444444444444444444444444444")
    print(docs[3].page_content)
    print("\n\n55555555555555555555555555555555555555555555555")
    print(docs[4].page_content)
    print("\n\n66666666666666666666666666666666666666666666666")
    print(docs[5].page_content)
    print("\n\n7777777777777777777777777777777777777777777777777777")
    print(docs[6].page_content)
    print("\n\n8888888888888888888888888888888888888888888888")
    print(docs[7].page_content)
    print("\n\n--------------------------------------")
    context = "\n\n".join(doc.page_content for doc in docs)
    print(context)
    context = docs
    print(context)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [67]:
answer_question("Who went to Mahchester University?", [])

[Document(id='1081e642-5b42-4f4f-bf83-661f695db904', metadata={'source': 'C:\\Users\\ME36352\\MyFolder\\AI\\Repos\\AIRepo\\week5\\knowledge-base\\employees\\Alex Thomson.md', 'doc_type': 'employees'}, page_content='# HR Record\n\n# Alex Thomson\n\n## Summary\n- **Date of Birth:** March 15, 1995\n- **Job Title:** Sales Development Representative (SDR)\n- **Location:** Austin, Texas\n- **Current Salary:** $65,000  \n\n## Insurellm Career Progression\n- **November 2022** - Joined Insurellm as a Sales Development Representative. Alex Thomson quickly adapted to the team, demonstrating exceptional communication and rapport-building skills.\n- **January 2023** - Promoted to Team Lead for special projects due to Alex\'s initiative in driving B2B customer outreach programs.  \n- **August 2023** - Developed a training module for new SDRs at Insurellm, enhancing onboarding processes based on feedback and strategies that Alex Thomson pioneered.  \n- **Current** - Continues to excel in the role, le

'Based on the information I have, Jessica Liu attended the University of Manchester.'

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!